<a href="https://colab.research.google.com/github/chaitanya731-spec/automated-question-answering/blob/main/Automated_Question_Answering_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automated Question Answering using Transformer Model

This notebook implements an **Automated Question Generation + Answering system** using pretrained Transformer models from Hugging Face.

**What this notebook does:**
1. Loads a pretrained **Extractive QA model** (DistilBERT fine-tuned on SQuAD) to answer questions from a passage.
2. Loads a pretrained **Question Generation (QG) model** (T5) to automatically generate questions from a passage.
3. Extracts candidate answers (key phrases/entities) from the passage using spaCy.
4. Combines QG + QA to produce **question–answer pairs automatically** from any passage you provide.
5. Provides an **interactive input section** where you paste your own passage and get generated Q&A pairs, or ask your own question.

> Runtime: This notebook works fine on Colab's free CPU runtime, but a GPU runtime (Runtime > Change runtime type > GPU) will make it noticeably faster.

## 1. Setup — Install Dependencies

In [1]:
!pip install -q transformers sentencepiece spacy torch
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import torch
import spacy
import textwrap
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

device = 0 if torch.cuda.is_available() else -1
print("Using GPU" if device == 0 else "Using CPU")

Using GPU


## 2. Load Pretrained Models

- **QA model**: `distilbert-base-cased-distilled-squad` — extractive question answering (finds the answer span inside the passage).
- **QG model**: `valhalla/t5-base-qg-hl` — sequence-to-sequence question generation, conditioned on a highlighted answer span.

In [4]:
from transformers import AutoModelForQuestionAnswering

qa_model_name = "distilbert-base-cased-distilled-squad"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)
if device == 0:
    qa_model = qa_model.to("cuda")


def qa_pipeline(question, context):
    """Mimic the old transformers QA pipeline's output: given a question and
    a context passage, return {'answer': str, 'score': float}."""
    inputs = qa_tokenizer(
        question, context, return_tensors="pt", truncation=True, max_length=384
    )
    if device == 0:
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]
    start_idx = int(torch.argmax(start_logits))
    end_idx = int(torch.argmax(end_logits))
    if end_idx < start_idx:
        end_idx = start_idx

    input_ids = inputs["input_ids"][0]
    answer = qa_tokenizer.decode(
        input_ids[start_idx:end_idx + 1], skip_special_tokens=True
    )

    start_prob = torch.softmax(start_logits, dim=0)[start_idx]
    end_prob = torch.softmax(end_logits, dim=0)[end_idx]
    score = float(start_prob * end_prob)

    return {"answer": answer.strip(), "score": score}


# Question Generation model (T5, answer-aware / highlight based)
qg_model_name = "valhalla/t5-base-qg-hl"
qg_tokenizer = AutoTokenizer.from_pretrained(qg_model_name)
qg_model = AutoModelForSeq2SeqLM.from_pretrained(qg_model_name)
if device == 0:
    qg_model = qg_model.to("cuda")

# spaCy for candidate-answer extraction (entities + noun chunks)
nlp = spacy.load("en_core_web_sm")

print("Models loaded successfully.")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Models loaded successfully.


## 3. Helper Functions

In [5]:
def extract_candidate_answers(passage, max_candidates=8):
    """Extract candidate answer phrases from a passage using named entities
    and noun chunks (deduplicated, longer/more informative phrases preferred)."""
    doc = nlp(passage)
    candidates = []

    for ent in doc.ents:
        candidates.append(ent.text.strip())

    for chunk in doc.noun_chunks:
        text = chunk.text.strip()
        if len(text.split()) <= 4 and text.lower() not in [c.lower() for c in candidates]:
            candidates.append(text)

    # Deduplicate while preserving order
    seen = set()
    unique_candidates = []
    for c in candidates:
        key = c.lower()
        if key not in seen and len(c) > 1:
            seen.add(key)
            unique_candidates.append(c)

    return unique_candidates[:max_candidates]


def generate_question(passage, answer):
    """Generate a question for a given answer span inside the passage,
    using the highlight-based T5 QG model."""
    if answer not in passage:
        return None

    highlighted = passage.replace(answer, f"<hl> {answer} <hl>", 1)
    input_text = f"generate question: {highlighted}"

    inputs = qg_tokenizer.encode(
        input_text, return_tensors="pt", truncation=True, max_length=512
    )
    if device == 0:
        inputs = inputs.to("cuda")

    outputs = qg_model.generate(
        inputs,
        max_length=64,
        num_beams=4,
        early_stopping=True
    )
    question = qg_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return question.strip()


def generate_qa_pairs(passage, num_questions=5):
    """Full pipeline: extract candidate answers -> generate a question for
    each -> verify/refine the answer using the extractive QA model."""
    candidates = extract_candidate_answers(passage, max_candidates=num_questions * 2)
    qa_pairs = []
    seen_questions = set()

    for answer in candidates:
        if len(qa_pairs) >= num_questions:
            break

        question = generate_question(passage, answer)
        if not question or question.lower() in seen_questions:
            continue

        # Verify the answer by actually running the QA model on the
        # generated question against the passage (self-consistency check)
        result = qa_pipeline(question=question, context=passage)
        final_answer = result["answer"]
        confidence = round(result["score"], 3)

        qa_pairs.append({
            "question": question,
            "answer": final_answer,
            "confidence": confidence
        })
        seen_questions.add(question.lower())

    return qa_pairs


def display_qa_pairs(qa_pairs):
    if not qa_pairs:
        print("No question-answer pairs could be generated. Try a longer passage.")
        return
    for i, pair in enumerate(qa_pairs, 1):
        print(f"Q{i}: {pair['question']}")
        print(f"A{i}: {pair['answer']}  (confidence: {pair['confidence']})\n")

## 4. Demo with a Sample Passage

In [6]:
sample_passage = (
    "The Transformer is a deep learning architecture introduced in the paper "
    "'Attention Is All You Need' by Vaswani et al. in 2017. Unlike earlier "
    "sequence models such as RNNs and LSTMs, the Transformer relies entirely "
    "on a mechanism called self-attention to process input sequences in "
    "parallel rather than sequentially. This architecture became the "
    "foundation for large language models such as BERT and GPT, and it is "
    "widely used today for tasks like machine translation, text "
    "summarization, and question answering."
)

print("Sample Passage:\n")
print(textwrap.fill(sample_passage, width=100))
print("\nGenerated Question-Answer Pairs:\n")

sample_qa_pairs = generate_qa_pairs(sample_passage, num_questions=5)
display_qa_pairs(sample_qa_pairs)

Sample Passage:

The Transformer is a deep learning architecture introduced in the paper 'Attention Is All You Need'
by Vaswani et al. in 2017. Unlike earlier sequence models such as RNNs and LSTMs, the Transformer
relies entirely on a mechanism called self-attention to process input sequences in parallel rather
than sequentially. This architecture became the foundation for large language models such as BERT
and GPT, and it is widely used today for tasks like machine translation, text summarization, and
question answering.

Generated Question-Answer Pairs:

Q1: What is the name of the deep learning architecture introduced in 2017?
A1: The Transformer  (confidence: 0.522)

Q2: What is the name of the paper that introduced the Transformer?
A2: Attention Is All You Need  (confidence: 0.677)

Q3: Who wrote the paper 'Attention is All You Need'?
A3: Vaswani et al  (confidence: 0.508)

Q4: When was the paper 'Attention is All You Need' published?
A4: 2017  (confidence: 0.986)

Q5: What is th

## 5. Interactive Section — Provide Your Own Passage

Run the cell below, then paste/type your own passage when prompted. The model will automatically generate a set of questions along with their answers.

In [9]:
# @title Enter your passage and generate Q&A pairs { display-mode: "form" }
user_passage = "The Great Barrier Reef, located off the coast of Queensland, Australia, is the world's largest coral reef system, stretching over 2,300 kilometers. It is composed of nearly 3,000 individual reefs and 900 islands, making it visible even from outer space. The reef is home to an extraordinary diversity of marine life, including over 1,500 species of fish, 400 types of coral, and various species of sharks, rays, and sea turtles. Despite its ecological importance, the Great Barrier Reef faces significant threats from climate change, particularly rising ocean temperatures that cause coral bleaching. In 2016 and 2017, the reef experienced two consecutive mass bleaching events that severely damaged large sections of coral. Scientists and conservationists continue to work on strategies to protect this UNESCO World Heritage Site, including reducing carbon emissions and improving water quality along the coastline."  # @param {type:"string"}
number_of_questions = 6  # @param {type:"slider", min:1, max:10, step:1}

print("Passage:\n")
print(textwrap.fill(user_passage, width=100))
print("\nGenerated Question-Answer Pairs:\n")

user_qa_pairs = generate_qa_pairs(user_passage, num_questions=number_of_questions)
display_qa_pairs(user_qa_pairs)

Passage:

The Great Barrier Reef, located off the coast of Queensland, Australia, is the world's largest coral
reef system, stretching over 2,300 kilometers. It is composed of nearly 3,000 individual reefs and
900 islands, making it visible even from outer space. The reef is home to an extraordinary diversity
of marine life, including over 1,500 species of fish, 400 types of coral, and various species of
sharks, rays, and sea turtles. Despite its ecological importance, the Great Barrier Reef faces
significant threats from climate change, particularly rising ocean temperatures that cause coral
bleaching. In 2016 and 2017, the reef experienced two consecutive mass bleaching events that
severely damaged large sections of coral. Scientists and conservationists continue to work on
strategies to protect this UNESCO World Heritage Site, including reducing carbon emissions and
improving water quality along the coastline.

Generated Question-Answer Pairs:

Q1: Where is the Great Barrier Reef lo

### 5.1 (Optional) Ask Your Own Question About the Passage

If you'd rather ask a specific question yourself instead of auto-generating questions, use the cell below.

In [8]:
# @title Ask a specific question about the passage above { display-mode: "form" }
your_question = "Type your question here"  # @param {type:"string"}

result = qa_pipeline(question=your_question, context=user_passage)
print(f"Q: {your_question}")
print(f"A: {result['answer']}  (confidence: {round(result['score'], 3)})")

Q: Type your question here
A: Paste your passage  (confidence: 0.22)


## 6. Notes & Possible Extensions

- **Better QG quality**: try `valhalla/t5-base-qg-hl` alternatives like `iarfmoose/t5-base-question-generator`, or fine-tune your own T5/BART model on SQuAD.
- **Better answer extraction**: replace the spaCy noun-chunk/entity heuristic with a dedicated answer-extraction model.
- **Evaluation**: compare generated Q&A pairs against a labeled dataset (e.g. SQuAD) using BLEU/ROUGE for question quality and EM/F1 for answer quality.
- **Deployment**: wrap `generate_qa_pairs` and `qa_pipeline` in a simple Streamlit or Gradio app for a shareable demo.